In [1]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../data/bpic12.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "case:REG_DATE_HR": "string",
        "case:REG_DATE_DAY": "string",
        "case:REG_DATE_MON": "string",
        "case:AMOUNT_REQ": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:AMOUNT_REQ,case:REG_DATE_DAY,case:REG_DATE_HR,case:REG_DATE_MON,concept:name,lifecycle:transition,org:resource,time_delta
0,173688,2011-10-01 00:38:44.546,20000.0,Saturday,12 AM,October,A_SUBMITTED,COMPLETE,112,0.000000
1,173688,2011-10-01 00:38:44.880,20000.0,Saturday,12 AM,October,A_PARTLYSUBMITTED,COMPLETE,112,0.334000
2,173688,2011-10-01 00:39:37.906,20000.0,Saturday,12 AM,October,A_PREACCEPTED,COMPLETE,112,53.026001
3,173688,2011-10-01 11:42:43.308,20000.0,Saturday,12 AM,October,A_ACCEPTED,COMPLETE,10862,39785.402344
4,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,O_SELECTED,COMPLETE,10862,145.934998
5,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,A_FINALIZED,COMPLETE,10862,0.000000
6,173688,2011-10-01 11:45:11.197,20000.0,Saturday,12 AM,October,O_CREATED,COMPLETE,10862,1.954000
7,173688,2011-10-01 11:45:11.380,20000.0,Saturday,12 AM,October,O_SENT,COMPLETE,10862,0.183000
8,173688,2011-10-10 11:33:03.668,20000.0,Saturday,12 AM,October,O_SENT_BACK,COMPLETE,11049,776872.312500
9,173688,2011-10-13 10:37:29.226,20000.0,Saturday,12 AM,October,A_REGISTERED,COMPLETE,10629,255865.562500


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:AMOUNT_REQ', 'case:REG_DATE_DAY', 'case:REG_DATE_HR', 'case:REG_DATE_MON', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 1315.95]                          0.4980     quantile_derived    
case:AMOUNT_REQ                continuous     case     yes    [3000.00, 40000.00]                      5000.0000  quantile_derived    
case:REG_DATE_DAY              categorical    case     yes    ['Friday', 'Monday', 'Saturday', ...]    N/A        

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[{'A_FINALIZED', 'O_SELECTED'},
 {'A_DECLINED', 'O_DECLINED'},
 {'A_ACTIVATED', 'A_APPROVED', 'A_REGISTERED', 'O_ACCEPTED'}]

In [13]:
engine.branching_sets

[{'A_CANCELLED',
  'A_FINALIZED',
  'O_CREATED',
  'O_SELECTED',
  'O_SENT',
  'O_SENT_BACK'},
 {'A_ACTIVATED',
  'A_APPROVED',
  'A_DECLINED',
  'A_REGISTERED',
  'O_ACCEPTED',
  'O_DECLINED'}]

### --- Experiments Generation ---

In [14]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic12-cf_seed777_experiments_ga_output.txt", console=False)

In [15]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [16]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

In [17]:
exp_df_mul, metadata_mul = ExperimentHandler.load("../experiments/cf_generated_experiments_multiple_desired")
print("Mined using parameters:", metadata_mul["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,210128,3,1,0,0.529653,0.449307,0.610000,0.668750,0.000000,...,0.181250,0.000000,0.056250,0.000000,0.112501,0.125000,0.0,0.000000,0.0,0.0
1,1,187256,3,1,0,0.417532,0.365064,0.470000,0.525000,0.000000,...,0.225000,0.000000,0.100000,0.200000,0.000000,0.125000,0.0,0.000000,0.0,0.0
2,1,211510,3,1,0,0.467168,0.474337,0.460000,0.606250,0.000000,...,0.384961,0.000000,0.134961,0.000000,0.269923,0.250000,0.0,0.000000,0.0,0.0
3,1,184360,3,1,0,0.503141,0.526282,0.480000,0.568750,0.000000,...,0.274757,0.000000,0.149757,0.000000,0.299513,0.125000,0.0,0.000000,0.0,0.0
4,1,204757,3,1,0,0.534647,0.569294,0.500000,0.625000,0.000000,...,0.250071,0.000000,0.125071,0.000000,0.250141,0.125000,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,49,195482,14,1,2,0.638491,0.536981,0.740000,0.798214,0.067742,...,1.061532,0.064516,0.425587,0.466667,0.384508,0.571429,0.0,0.011512,0.0,0.0
308,49,183292,14,1,2,0.666985,0.560637,0.773333,0.778571,0.077419,...,1.040226,0.064516,0.439996,0.466667,0.413324,0.535714,0.0,0.000000,0.0,0.0
309,49,196861,14,1,2,0.607704,0.438741,0.776667,0.758929,0.148387,...,0.939595,0.064516,0.375079,0.400000,0.350158,0.500000,0.0,0.000000,0.0,0.0
310,49,185452,14,1,2,0.652901,0.529135,0.776667,0.825000,0.116129,...,1.092439,0.129032,0.356264,0.533333,0.179195,0.607143,0.0,0.000000,0.0,0.0


In [21]:
results_mul = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_multiple_desired_seed777",
    exp_df=exp_df_mul,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [22]:
results_mul

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,177657,13,2,6,0.574527,0.567235,0.581818,0.6975,0.187500,...,0.795825,0.187500,0.258325,0.181818,0.334832,0.35,0.000000,0.442557,0.000000,0.500000
1,0,192659,13,2,6,0.570138,0.540276,0.600000,0.6600,0.312500,...,0.977706,0.312500,0.315206,0.272727,0.357684,0.35,0.000000,0.473435,0.000000,0.500000
2,0,189589,13,2,6,0.485834,0.439849,0.531818,0.6000,0.187500,...,0.859884,0.187500,0.322384,0.272727,0.372041,0.35,0.000000,0.441304,0.000000,0.500000
3,0,191566,13,2,6,0.554818,0.582362,0.527273,0.6125,0.193750,...,0.785969,0.187500,0.298469,0.181818,0.415119,0.30,0.000000,0.433888,0.000000,0.500000
4,0,198735,13,2,6,0.419942,0.358066,0.481818,0.4925,0.190625,...,0.926976,0.187500,0.000000,0.000000,0.000000,0.00,0.739476,0.747602,0.999999,0.999999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,48,189055,13,4,6,0.552076,0.467788,0.636364,0.6925,0.052632,...,0.910228,0.052632,0.357597,0.363636,0.351557,0.50,0.000000,0.701528,0.000000,0.750000
292,48,214226,13,4,6,0.641570,0.574048,0.709091,0.7600,0.052632,...,0.836469,0.052632,0.333837,0.272727,0.394947,0.45,0.000000,0.700672,0.000000,0.750000
293,48,205316,13,4,6,0.559664,0.546601,0.572727,0.7225,0.052632,...,0.685175,0.052632,0.232544,0.090909,0.374178,0.40,0.000000,0.631608,0.000000,0.749999
294,49,207434,12,4,2,0.459287,0.386755,0.531818,0.5300,0.144444,...,0.403480,0.138889,0.000000,0.000000,0.000000,0.00,0.264591,0.486295,0.250000,0.500000


### --- Cleanup ---

In [23]:
sys.stdout = original_stdout
log_file.close()